# Find Best Model


In [296]:
from typing import Callable
from types import SimpleNamespace
from collections import defaultdict
import os
import glob
import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go

from meta import load_meta
from data_module import NPZTensorDataModule
from model import MultiLayerLeakyReLUModel

## Args


### Demo


In [297]:
# args = SimpleNamespace(
#     # io
#     logs_path="data/logs",
#     experiment_name="demo_experiment",
# )

### laser-pulse-shaping-astra-sim


In [298]:
args = SimpleNamespace(
    # io
    data_path="data/laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
    logs_path="data/logs",
    experiment_name="laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
    # data
    batch_size=64,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
)

In [299]:
rows: list[dict] = []

for run_path in glob.glob(os.path.join(args.logs_path, args.experiment_name, "*")):
    meta = load_meta(run_path)
    rows.append(meta)

table = pd.DataFrame(rows)

assert all(table["data_path"] == args.data_path)

table["max_width"] = table["hidden_layer_sizes"].apply(lambda x: max(x))
table["depth"] = table["hidden_layer_sizes"].apply(lambda x: len(x))


table["best_val_train_loss_ratio"] = table["best_val_loss"] / table["best_train_loss"]

table["log10(trainable_params)"] = table["trainable_params"].apply(
    lambda x: np.log10(x)
)
table["log10(best_val_loss)"] = table["best_val_loss"].apply(lambda x: np.log10(x))
table["log10(best_train_loss)"] = table["best_train_loss"].apply(lambda x: np.log10(x))
table["log10(learning_rate)"] = table["learning_rate"].apply(lambda x: np.log10(x))
table["log2(batch_size)"] = table["batch_size"].apply(lambda x: np.log2(x))
table["log10(best_step)"] = table["best_step"].apply(lambda x: np.log10(x))


table

,data_path,logs_path,experiment_name,version,batch_size,num_workers,persistent_workers,pin_memory,hidden_layer_sizes,leaky_relu_factor,...,best_train_loss,max_width,depth,best_val_train_loss_ratio,log10(trainable_params),log10(best_val_loss),log10(best_train_loss),log10(learning_rate),log2(batch_size),log10(best_step)
0,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-09-Nd6QJXHA,128,8,True,True,"[10, 10, 10, 10]",0.1,...,0.063579,10,4,0.999171,2.871573,-1.197048,-1.196688,-4.0,7.0,5.999899
1,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-12-CZho8sGU,128,8,True,True,"[100, 100, 100]",0.1,...,0.002301,100,3,1.844048,4.385678,-2.372386,-2.638158,-2.0,7.0,5.621450
2,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-10-oD39wBe9,128,8,True,True,"[50, 50, 50]",0.1,...,0.006820,50,3,1.171207,3.854549,-2.097594,-2.166228,-3.0,7.0,5.999765
3,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-12-9zGLfPKv,128,8,True,True,"[100, 100, 100, 100]",0.1,...,0.006717,100,4,1.422918,4.536609,-2.019625,-2.172805,-3.0,7.0,5.870027
4,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-13-LteTUXnE,128,8,True,True,"[200, 200]",0.1,...,0.050154,200,2,1.035673,4.684881,-1.284469,-1.299692,-4.0,7.0,5.999996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-14-86aUFbtc,128,8,True,True,"[200, 200, 200]",0.1,...,0.001059,200,3,4.344096,4.947453,-2.337276,-2.975176,-2.0,7.0,5.825105
317,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-12-f43xXK96,128,8,True,True,"[100, 100, 100, 100]",0.1,...,0.012806,100,4,1.283768,4.536609,-1.784110,-1.892596,-3.0,7.0,5.608301
318,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-11-GdwLdUDA,128,8,True,True,"[50, 50, 50, 50]",0.1,...,0.001612,50,4,1.462059,3.986951,-2.627723,-2.792688,-2.0,7.0,5.997854
319,data/laser-pulse-shaping-astra-sim-v11-ln-emit...,data/logs,laser-pulse-shaping-astra-sim-v11-ln-emit-norm...,2025-12-13-13-34-12-4fEnBtEs,128,8,True,True,"[100, 100]",0.1,...,0.001206,100,2,2.102846,4.152411,-2.595976,-2.918783,-1.0,7.0,5.264525


In [300]:
px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="log2(batch_size)",
).show()

px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="best_val_train_loss_ratio",
    color_continuous_scale="RdBu",  # red–blue color scale
    range_color=[0, 2],  # set colorbar limits
).show()

px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

px.scatter(
    table,
    x="max_width",
    y="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

px.scatter(
    table,
    x="depth",
    y="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

## Best Model Overall


In [301]:
best = table.iloc[table["log10(best_val_loss)"].argmin()]
best

data_path                                                data/laser-pulse-shaping-astra-sim-v11-ln-emit...
logs_path                                                                                        data/logs
experiment_name                                          laser-pulse-shaping-astra-sim-v11-ln-emit-norm...
version                                                                       2025-12-13-13-34-11-ANimeJpN
batch_size                                                                                             128
num_workers                                                                                              8
persistent_workers                                                                                    True
pin_memory                                                                                            True
hidden_layer_sizes                                                                    [50, 50, 50, 50, 50]
leaky_relu_factor                    

In [302]:
best_path = f"{args.logs_path}/{args.experiment_name}/_best"

if os.path.islink(best_path):
    os.unlink(best_path)

os.symlink(best["version"], best_path, target_is_directory=True)

## Inspect Best Model


In [303]:
selected_version = best["version"]
selected_version

'2025-12-13-13-34-11-ANimeJpN'

In [304]:
selected_version_path = os.path.join(
    args.logs_path, args.experiment_name, selected_version
)
selected_ckpt_path = os.path.join(selected_version_path, "best.ckpt")
selected_version_path

'data/logs/laser-pulse-shaping-astra-sim-v11-ln-emit-normalized/2025-12-13-13-34-11-ANimeJpN'

### load data


In [305]:
def _partial_apply(
    x: torch.Tensor, func: Callable[[torch.Tensor], torch.Tensor], slc: list[slice]
) -> torch.Tensor:
    y = x.clone()
    y[*slc] = func(x[*slc])
    return y

In [306]:
data = NPZTensorDataModule(
    path=args.data_path,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    persistent_workers=args.persistent_workers,
    pin_memory=args.pin_memory,
)
data.setup("test")
test_data_loader = data.test_dataloader()

data_meta = load_meta(args.data_path)
x_size = data_meta["input_size"]
y_size = data_meta["output_size"]
x_params = data_meta["columns"][:x_size]
y_params = data_meta["columns"][x_size:]
x_std = (
    torch.tensor(data_meta["normalization_original_std"][:x_size])
    if "normalization_original_std" in data_meta
    else torch.ones(x_size)
)
x_mean = (
    torch.tensor(data_meta["normalization_original_mean"][:x_size])
    if "normalization_original_mean" in data_meta
    else torch.zeros(x_size)
)
y_std = (
    torch.tensor(data_meta["normalization_original_std"][x_size:])
    if "normalization_original_std" in data_meta
    else torch.ones(y_size)
)
y_mean = (
    torch.tensor(data_meta["normalization_original_mean"][x_size:])
    if "normalization_original_mean" in data_meta
    else torch.zeros(y_size)
)
x_outer_back_trafo = lambda x: x
y_outer_back_trafo = (
    (lambda y: _partial_apply(y, np.exp, [slice(None), slice(0, 2)]))
    if "ln-emit" in args.data_path
    else lambda y: y
)

assert len(x_params) == x_size
assert len(y_params) == y_size

pd.Series(data_meta)

author                                                                  kwasniok
date                                            2025-12-13T11:59:14.048542+00:00
description                    Dataset for surrogate modeling of ASTRA simula...
data_path                      data/laser-pulse-shaping-astra-sim-v11-ln-emit...
input_size                                                                    36
output_size                                                                    4
columns                        [latent vector component 00, latent vector com...
normalization_original_mean    [-0.15776555973263068, -0.1333923397207948, -0...
normalization_original_std     [0.9752264548504911, 0.8618616902325862, 0.862...
dtype: object

### load model


In [307]:
model = MultiLayerLeakyReLUModel.load_from_checkpoint(selected_ckpt_path)
model.eval()

MultiLayerLeakyReLUModel(
  (loss_fn): MSELoss()
  (sequence): Sequential(
    (0): Linear(in_features=36, out_features=50, bias=True)
    (1): LeakyReLU(negative_slope=0.1)
    (2): Linear(in_features=50, out_features=50, bias=True)
    (3): LeakyReLU(negative_slope=0.1)
    (4): Linear(in_features=50, out_features=50, bias=True)
    (5): LeakyReLU(negative_slope=0.1)
    (6): Linear(in_features=50, out_features=50, bias=True)
    (7): LeakyReLU(negative_slope=0.1)
    (8): Linear(in_features=50, out_features=50, bias=True)
    (9): LeakyReLU(negative_slope=0.1)
    (10): Linear(in_features=50, out_features=4, bias=True)
  )
)

### evaluate model


In [308]:
epsilon = 1e-12

In [309]:
def back_trafo_x(x):
    return x_outer_back_trafo(x * x_std + x_mean)


def back_trafo_y(y):
    return y_outer_back_trafo(y * y_std + y_mean)

In [310]:
results = defaultdict(list)
with torch.no_grad():
    for x, y in test_data_loader:
        y_pred = model(x)

        # transform back
        x = back_trafo_x(x)
        y = back_trafo_y(y)
        y_pred = back_trafo_y(y_pred)

        results["x"].append(x)
        results["y"].append(y)
        results["y_pred"].append(y_pred)
        results["y_rel_err"].append(((y_pred - y) / (y)))

for k, v in results.items():
    results[k] = torch.cat(v).cpu().numpy()

results = (
    {k: results["x"][:, i] for i, k in enumerate(x_params)}
    | {k: results["y"][:, i] for i, k in enumerate(y_params)}
    | {f"{k} prediciton": results["y_pred"][:, i] for i, k in enumerate(y_params)}
    | {f"{k} rel. error": results["y_rel_err"][:, i] for i, k in enumerate(y_params)}
)

results = pd.DataFrame(results)

/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.



/tmp/ipykernel_831860/2308167257.py:5: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)



In [311]:
results.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
latent vector component 00,6150.0,-0.166736,0.969708,-3.457005,-0.753557,-0.142631,0.393356,3.573657
latent vector component 01,6150.0,-0.134150,0.856268,-4.025670,-0.694360,-0.102434,0.444629,3.016539
latent vector component 02,6150.0,-0.505440,0.862224,-4.072786,-0.990210,-0.422293,0.051647,2.792728
latent vector component 03,6150.0,-0.158208,0.896733,-2.839366,-0.779514,-0.262667,0.353179,3.392973
latent vector component 04,6150.0,-0.063380,0.836408,-3.766775,-0.553619,-0.070425,0.440164,3.894987
latent vector component 05,6150.0,-0.319460,0.825468,-3.773775,-0.828532,-0.250821,0.215147,2.640937
latent vector component 06,6150.0,0.134927,0.666221,-2.293424,-0.283242,0.132246,0.548639,2.739383
latent vector component 07,6150.0,0.383431,0.818075,-2.737159,-0.140858,0.377324,0.874644,4.302205
latent vector component 08,6150.0,0.007847,0.778582,-4.241605,-0.411714,0.019457,0.457585,3.604238
latent vector component 09,6150.0,-0.041321,0.752374,-3.451249,-0.463562,-0.038041,0.397627,3.261094


In [312]:
for y in y_params:
    y_rel_err = f"{y} rel. error"
    fig = px.scatter(
        results,
        x=y,
        y=y_rel_err,
        log_x=True,
        # log_y=True,
        title=f"relative error for {y} predictions",
        labels={y: y, y_rel_err: "rel. err."},
    )
    fig.add_hline(
        y=results[y_rel_err].mean(),
        line_dash="dash",
        line_color="black",
        name="mean rel. err.",
    )
    # fake legend entry for mean line
    fig.add_trace(
        go.Scatter(
            x=[results[y].min(), results[y].min()],
            y=[results[y_rel_err].mean(), results[y_rel_err].mean()],
            mode="lines",
            line=dict(dash="dash", color="black"),
            name="mean rel. err.",
        )
    )
    for q in [0.25, 0.5, 0.75]:
        fig.add_hline(
            y=results[y_rel_err].quantile(q),
            line_dash="dot",
            line_color="gray",
            name=f"{q*100}% quantile",
        )
    # fake legend entry for quantile lines
    fig.add_trace(
        go.Scatter(
            x=[results[y].min(), results[y].min()],
            y=[
                results[y_rel_err].quantile(0.5),
                results[y_rel_err].quantile(0.5),
            ],
            mode="lines",
            line=dict(dash="dot", color="gray"),
            name=f"quantile rel. err.",
        )
    )
    fig.update_traces(marker=dict(size=3, opacity=0.5))
    fig.show()

    fig.write_image(f"{selected_version_path}/eval/rel_err_{y}.pdf")